# 🎬 DARK STICKMAN STUDIO — Google Colab
## Pipeline F01: Roteiro → MP4 Final

**DARK CREATIVE LAB** | Canal faceless automatizado para YouTube

---
### ✅ O que este notebook faz:
1. Instala todas as dependências
2. Configura o ambiente (TTS, FFmpeg, GPU)
3. Executa o pipeline completo F01
4. Baixa o vídeo MP4 final

### 💡 Vantagens do Colab:
- **GPU T4 gratuita** (15GB VRAM) — roda modelos locais
- **Kokoro TTS** local sem proxy
- **Piper TTS** com download direto
- Sem restrições de rede para HuggingFace

> `ai_generated: True | ai_disclosure: Produzido com DARK STUDIO AAA — DARK CREATIVE LAB`

---
## 🔧 CÉLULA 1 — Diagnóstico de Hardware

In [ ]:
# CÉLULA 1 — DIAGNÓSTICO DE HARDWARE
# =====================================================================
import subprocess, platform, sys, os, shutil

print('=' * 60)
print('  DARK STICKMAN STUDIO — Diagnóstico de Hardware')
print('=' * 60)

# Sistema
print(f'\n🖥️  Sistema: {platform.system()} {platform.release()}')
print(f'🐍 Python: {sys.version.split()[0]}')

# CPU
try:
    cpu = subprocess.check_output(['lscpu'], text=True)
    for line in cpu.split('\n'):
        if 'Model name' in line or 'CPU(s)' in line:
            print(f'⚙️  CPU: {line.split(":")[1].strip()}')
            break
except:
    pass

# RAM
try:
    with open('/proc/meminfo') as f:
        for line in f:
            if 'MemTotal' in line:
                ram_kb = int(line.split()[1])
                print(f'💾 RAM: {ram_kb/1024/1024:.1f} GB')
                break
except:
    pass

# GPU
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                                        '--format=csv,noheader'], text=True).strip()
    print(f'🎮 GPU: {gpu_info}')
    GPU_AVAILABLE = True
except:
    print('❌ GPU: não disponível (usando CPU)')
    GPU_AVAILABLE = False

# CUDA
try:
    import torch
    print(f'🔥 PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
except ImportError:
    print('⚠️  PyTorch: não instalado ainda')

# Disco
disk = shutil.disk_usage('/')
print(f'💿 Disco: {disk.free/1e9:.1f} GB livres de {disk.total/1e9:.1f} GB')

# FFmpeg
ffmpeg_path = shutil.which('ffmpeg')
print(f'🎞️  FFmpeg: {"✅ " + ffmpeg_path if ffmpeg_path else "❌ não instalado"}')

print('\n' + '=' * 60)
print(f'✅ Ambiente: {"GPU disponível" if GPU_AVAILABLE else "CPU only"}')
print('=' * 60)

---
## 📦 CÉLULA 2 — Instalação de Dependências

In [ ]:
# CÉLULA 2 — INSTALAÇÃO (executa uma vez)
# =====================================================================
import subprocess, sys

def instalar(pacote, label=None):
    label = label or pacote
    print(f'📦 Instalando {label}...', end=' ')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pacote, '-q'], capture_output=True)
    print('✅' if r.returncode == 0 else f'❌ {r.stderr[-100:]}')

# Core
instalar('pillow', 'Pillow')
instalar('requests')

# TTS
print('\n🎙️  Instalando engines TTS...')
instalar('kokoro', 'Kokoro TTS (local, 82M)')
instalar('soundfile', 'SoundFile')
instalar('edge-tts', 'Edge TTS (Microsoft)')
instalar('gtts', 'gTTS')

# Legendas / NLP
print('\n📝 Instalando legendas e NLP...')
instalar('faster-whisper', 'Faster-Whisper (transcrição)')

# Gradio client (Kokoro Space)
instalar('gradio_client', 'Gradio Client (Kokoro Space)')

# Upscaling
instalar('realesrgan', 'Real-ESRGAN (upscaling)')

print('\n✅ Instalação concluída!')
print('⚠️  Se houver erros, execute a célula novamente.')

---
## 📁 CÉLULA 3 — Clone do Repositório DARK STICKMAN

In [ ]:
# CÉLULA 3 — CLONE DO REPOSITÓRIO
# =====================================================================
import os, subprocess

REPO_URL = 'https://github.com/Jeffersonjazz/dark-stickman-studio.git'
REPO_DIR = '/content/dark-stickman-studio'

if os.path.exists(REPO_DIR):
    print('📁 Repositório já clonado. Atualizando...')
    subprocess.run(['git', 'pull'], cwd=REPO_DIR)
else:
    print(f'📥 Clonando {REPO_URL}...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print(f'✅ Diretório: {os.getcwd()}')

# Adiciona ao PATH do Python
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('📋 Estrutura do projeto:')
for d in ['core', 'pipeline', 'config', 'api']:
    p = os.path.join(REPO_DIR, d)
    if os.path.exists(p):
        files = os.listdir(p)
        print(f'  {d}/ → {len(files)} arquivos')

---
## 🔑 CÉLULA 4 — Configuração de Chaves de API (Opcional)

In [ ]:
# CÉLULA 4 — CONFIGURAÇÃO DE CHAVES (OPCIONAL)
# =====================================================================
# Configure suas chaves aqui para desbloquear serviços premium.
# Todas são OPCIONAIS — o pipeline funciona sem elas usando fallbacks.

import os

# ─── HUGGING FACE (gratuito) ───────────────────────────────────────
# Cria conta em huggingface.co → Settings → Access Tokens → New Token
# Desbloqueia: MMS-TTS-POR (PT-BR nativo), SpeechT5
os.environ['HF_TOKEN'] = ''  # ← coloque seu token aqui: hf_xxx

# ─── ELEVENLABS (gratuito: 10k chars/mês) ──────────────────────────
# Cria conta em elevenlabs.io → Profile → API Key
# Melhor qualidade de voz PT-BR disponível
os.environ['ELEVENLABS_API_KEY'] = ''  # ← sk_xxx

# ─── REPLICATE (opcional, ~$0.02/seg de vídeo) ────────────────────
# Para geração de vídeos realistas via API
os.environ['REPLICATE_API_TOKEN'] = ''  # ← r8_xxx

# ─── STATUS ───────────────────────────────────────────────────────
apis = {
    'HF_TOKEN': os.environ.get('HF_TOKEN', ''),
    'ELEVENLABS_API_KEY': os.environ.get('ELEVENLABS_API_KEY', ''),
    'REPLICATE_API_TOKEN': os.environ.get('REPLICATE_API_TOKEN', ''),
}

for k, v in apis.items():
    status = '✅ Configurado' if v else '⚠️  Não configurado (fallback)'
    print(f'{k}: {status}')

print('\n💡 O pipeline sempre funciona mesmo sem chaves (silêncio / FFmpeg fallback)')

---
## 🎙️ CÉLULA 5 — Download do Modelo Piper TTS (PT-BR)

In [ ]:
# CÉLULA 5 — PIPER TTS (PT-BR) — melhor qualidade local
# =====================================================================
import subprocess, os, sys

MODEL_DIR = '/content/models/piper'
os.makedirs(MODEL_DIR, exist_ok=True)

# Download do executável Piper
PIPER_URL = 'https://github.com/rhasspy/piper/releases/download/2023.11.14-2/piper_linux_x86_64.tar.gz'
PIPER_BIN = '/content/piper/piper'

if not os.path.exists(PIPER_BIN):
    print('📥 Baixando Piper TTS...')
    subprocess.run(['wget', '-q', PIPER_URL, '-O', '/tmp/piper.tar.gz'])
    subprocess.run(['tar', '-xzf', '/tmp/piper.tar.gz', '-C', '/content/'])
    print('✅ Piper instalado')
else:
    print('✅ Piper já instalado')

# Download modelo PT-BR (Faber - alta qualidade)
MODEL_FILES = [
    ('pt_BR-faber-medium.onnx', 'https://huggingface.co/rhasspy/piper-voices/resolve/main/pt/pt_BR/faber/medium/pt_BR-faber-medium.onnx'),
    ('pt_BR-faber-medium.onnx.json', 'https://huggingface.co/rhasspy/piper-voices/resolve/main/pt/pt_BR/faber/medium/pt_BR-faber-medium.onnx.json'),
]

for filename, url in MODEL_FILES:
    path = os.path.join(MODEL_DIR, filename)
    if not os.path.exists(path):
        print(f'📥 Baixando {filename}...')
        subprocess.run(['wget', '-q', url, '-O', path])
        print(f'✅ {filename}')
    else:
        print(f'✅ {filename} (já existe)')

# Configura variáveis de ambiente para piper_tts.py
os.environ['PIPER_BIN'] = PIPER_BIN
os.environ['PIPER_MODEL'] = os.path.join(MODEL_DIR, 'pt_BR-faber-medium.onnx')

# Teste rápido
if os.path.exists(PIPER_BIN):
    result = subprocess.run(
        f'echo "Olá, Dark Stickman Studio está funcionando!" | {PIPER_BIN} --model {MODEL_DIR}/pt_BR-faber-medium.onnx --output_file /tmp/teste.wav 2>/dev/null',
        shell=True
    )
    print(f'\n🎙️  Teste Piper: {"✅ OK" if result.returncode == 0 else "❌ Falhou"}')

---
## ✍️ CÉLULA 6 — Roteiro de Exemplo

In [ ]:
# CÉLULA 6 — ROTEIRO (edite aqui!)
# =====================================================================
# Configure o vídeo que você quer produzir.
# O pipeline gera automaticamente: narração, animação, legendas, MP4.

CANAL_CONFIG = {
    # ─── Identificação ───────────────────────────────────────────────
    'id': 'dark_documentary',            # ← campo obrigatório para o pipeline
    'canal_id': 'dark_documentary',      # ← alias para compatibilidade
    'titulo': 'A História Secreta dos Stickmen',
    'categoria': 'documentario',         # documentario | natureza | historia | tecnologia | misterio

    # ─── Visual ──────────────────────────────────────────────────────
    'resolucao': '1920x1080',
    'fps': 30,
    'formato': '16:9',                   # 16:9 | 9:16 (Shorts)
    'color_grade': 'documentario',       # documentario | suspense | natureza | tecnologia

    # ─── Áudio ───────────────────────────────────────────────────────
    'voz': 'pt_BR-faber-medium',
    'velocidade_fala': 1.0,
    'nicho': 'documentario',

    # ─── Features ────────────────────────────────────────────────────
    'video_realista': False,             # True = usa DARK VIDEO STUDIO (GPU)
    'video_preset': 'documentario',
    'watermark': 'DARK STUDIO AAA',
}

# Roteiro com cenas
ROTEIRO = {
    'titulo': CANAL_CONFIG['titulo'],
    'canal': CANAL_CONFIG['id'],
    'hook': 'Você nunca imaginou que por trás de uma simples animação de palito...'
             ' existia um universo secreto.',
    'cenas': [
        {
            'titulo': 'O Início',
            'narracao': 'Em 2006, um jovem animador criou o primeiro stickman que lutava contra seu próprio criador. '
                       'O vídeo teve 50 milhões de visualizações em uma semana.',
            'duracao': 6,
            'tipo': 'abertura',
            'prompt_visual': 'cinematic shot of a simple stick figure drawing coming to life on white paper, dramatic lighting',
        },
        {
            'titulo': 'A Revolução',
            'narracao': 'O que começou como uma brincadeira se transformou em um gênero inteiro de animação. '
                       'Hoje, canais de stickman acumulam bilhões de visualizações no YouTube.',
            'duracao': 7,
            'tipo': 'desenvolvimento',
            'prompt_visual': 'montage of viral stick figure animation videos on youtube, trending content creator',
        },
        {
            'titulo': 'O Segredo',
            'narracao': 'Mas qual é o segredo por trás do sucesso dos canais de stickman? '
                       'A resposta está na simplicidade que libera a imaginação.',
            'duracao': 6,
            'tipo': 'desenvolvimento',
            'prompt_visual': 'close up of hand drawing stick figures on paper, artistic and creative process',
        },
        {
            'titulo': 'O Futuro',
            'narracao': 'Com inteligência artificial e automação, o futuro das animações faceless é ainda mais promissor. '
                       'Um único criador pode agora produzir conteúdo para dezenas de canais.',
            'duracao': 7,
            'tipo': 'conclusao',
            'prompt_visual': 'futuristic AI animation studio with multiple screens showing stick figure animations',
        },
    ],
    'cta': 'Se você chegou até aqui, deixe seu like e se inscreva para mais histórias incríveis!',
}

print('✅ Roteiro configurado!')
print(f'📹 Título: {ROTEIRO["titulo"]}')
print(f'🎬 Cenas: {len(ROTEIRO["cenas"])}')
total_narr = sum(c['duracao'] for c in ROTEIRO['cenas'])
print(f'⏱️  Duração estimada: ~{total_narr + 15}s')
print(f'🎨 Categoria: {CANAL_CONFIG["categoria"]}')
print(f'🆔 Canal ID: {CANAL_CONFIG["id"]}')


---
## 🚀 CÉLULA 7 — Execução do Pipeline F01

In [ ]:
# CÉLULA 7 — EXECUTAR PIPELINE F01
# =====================================================================
import sys, os

# Garante que o repo está no path
REPO_DIR = '/content/dark-stickman-studio'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from pipeline.f01_producao import produzir_video

print('🎬 Iniciando Pipeline F01...')
print('=' * 50)

resultado = produzir_video(
    roteiro=ROTEIRO,
    canal_config=CANAL_CONFIG,
    output_dir='/content/output',
    job_id='colab_job_01',
)

print('=' * 50)
print(f'\n📊 STATUS: {resultado["status"]}')

if resultado['status'] == 'ok':
    print(f'🎉 VÍDEO GERADO COM SUCESSO!')
    print(f'📁 Arquivo: {resultado["output"]}')
    import os
    size_mb = os.path.getsize(resultado['output']) / 1e6
    print(f'📦 Tamanho: {size_mb:.1f} MB')
else:
    print(f'⚠️  Erro: {resultado.get("erro", "desconhecido")}')


---
## 📥 CÉLULA 8 — Download e Preview do Vídeo

In [ ]:
# CÉLULA 8 — DOWNLOAD E PREVIEW
# =====================================================================
from IPython.display import HTML, display, Video
from google.colab import files
import os

OUTPUT_DIR = '/content/output'

# Lista todos os vídeos gerados
videos = []
for root, dirs, filenames in os.walk(OUTPUT_DIR):
    for f in filenames:
        if f.endswith(('.mp4', '.mkv')):
            videos.append(os.path.join(root, f))

if not videos:
    print('❌ Nenhum vídeo encontrado. Execute a Célula 7 primeiro.')
else:
    print(f'✅ {len(videos)} vídeo(s) encontrado(s):')
    for v in videos:
        size_mb = os.path.getsize(v) / 1e6
        print(f'  📹 {os.path.basename(v)} ({size_mb:.1f} MB)')

    # Preview do vídeo principal
    video_principal = sorted(videos)[-1]  # último gerado
    print(f'\n▶️  Preview: {os.path.basename(video_principal)}')

    display(Video(video_principal, width=720, embed=True))

    # Download
    print('\n💾 Fazendo download...')
    files.download(video_principal)

---
## 🎥 CÉLULA 9 — Geração de Vídeo Realista (GPU T4)

In [ ]:
# CÉLULA 9 — DARK VIDEO STUDIO (Opcional — requer GPU)
# =====================================================================
# Gera vídeos realistas usando modelos de IA.
# O Colab T4 (15GB VRAM) pode rodar modelos menores.

import sys, os
sys.path.insert(0, '/content/dark-stickman-studio')
os.chdir('/content/dark-stickman-studio')

from core.dark_video_studio import check_backends, generate_video

# Verifica backends disponíveis
print('🔍 Verificando backends de vídeo...')
status = check_backends()

for backend, info in status.items():
    disponivel = info.get('available', False)
    icon = '✅' if disponivel else '❌'
    print(f'  {icon} {backend}: {info.get("note", "")}')

# Geração de teste
print('\n🎬 Gerando vídeo de teste...')
result = generate_video(
    prompt='cinematic documentary shot of a stick figure hero standing on a mountain, dramatic sunset, epic composition',
    output_path='/content/output/video_realista_teste.mp4',
    preset='documentario',
    duration=5.0,
    verbose=True,
)

print(f'\n📊 Backend usado: {result["backend"]}')
print(f'⏱️  Tempo: {result.get("duration_s", 0):.1f}s')
print(f'💰 Custo: ${result.get("cost_usd", 0):.4f}')

if result['status'] == 'ok':
    from IPython.display import Video, display
    display(Video(result['output_path'], width=720, embed=True))

---
## 🎙️ CÉLULA 10 — Teste de Engines TTS

In [ ]:
# CÉLULA 10 — TESTE DE ENGINES TTS
# =====================================================================
import sys, os
sys.path.insert(0, '/content/dark-stickman-studio')
os.chdir('/content/dark-stickman-studio')

from core.cloud_tts import check_cloud_tts, synthesize_cloud
from IPython.display import Audio, display
from pathlib import Path

# Status dos engines
print('🎙️  Status dos engines TTS:\n')
status = check_cloud_tts()
for name, info in status.items():
    icon = '✅' if info['available'] else '❌'
    print(f'  {icon} {name}')
    print(f'     {info["note"]}')
    print(f'     💰 {info["cost"]}')
    print()

# Síntese de teste
TEXTO_TESTE = (
    'Olá! Eu sou o Dark Stickman Studio. '
    'Produzo vídeos automatizados para YouTube com inteligência artificial. '
    'Este é um teste de narração em português brasileiro.'
)

output_wav = Path('/content/output/teste_tts.wav')
output_wav.parent.mkdir(parents=True, exist_ok=True)

print(f'🎤 Sintetizando: "{TEXTO_TESTE[:50]}..."')
result = synthesize_cloud(TEXTO_TESTE, output_wav, verbose=True)

print(f'\n✅ Backend: {result["backend_used"]}')

if output_wav.exists() and output_wav.stat().st_size > 100:
    print('▶️  Reproduzindo:')
    display(Audio(str(output_wav)))
else:
    print('⚠️  Arquivo de áudio não gerado (silêncio fallback)')

---
## 📊 CÉLULA 11 — Produção em Batch (12 Canais)

In [ ]:
# CÉLULA 11 — BATCH PRODUCTION
# =====================================================================
# Produz vídeos para múltiplos canais automaticamente.
# Requer configuração de roteiros em config/canais/

import sys, os, json, glob
sys.path.insert(0, '/content/dark-stickman-studio')
os.chdir('/content/dark-stickman-studio')

from pipeline.f01_producao import produzir_video

CANAIS_DIR = 'config/canais'
canais = glob.glob(f'{CANAIS_DIR}/*.json')

if not canais:
    print('⚠️  Nenhum canal configurado em config/canais/')
    print('   Configure roteiros em JSON e execute novamente.')
else:
    print(f'🎬 Produzindo {len(canais)} canal(is)...')
    resultados = []

    for canal_file in canais[:3]:  # Máximo 3 por vez no Colab
        with open(canal_file) as f:
            config = json.load(f)

        canal_cfg = config.get('config', {})
        # Garante campo 'id' obrigatório
        if 'id' not in canal_cfg:
            canal_cfg['id'] = canal_cfg.get('canal_id', os.path.splitext(os.path.basename(canal_file))[0])
        canal_id = canal_cfg['id']

        print(f'\n  🎥 Canal: {canal_id}')

        roteiro = config.get('roteiro', {})
        job_id = f'batch_{canal_id}'

        result = produzir_video(
            roteiro=roteiro,
            canal_config=canal_cfg,
            output_dir=f'/content/output/{canal_id}',
            job_id=job_id,
        )
        resultados.append({'canal': canal_id, 'status': result['status'],
                          'output': result.get('output', '')})
        print(f'  {"✅" if result["status"] == "ok" else "❌"} {result["status"]}')

    # Resumo
    print('\n📊 RESUMO:')
    ok = sum(1 for r in resultados if r['status'] == 'ok')
    print(f'  ✅ Sucesso: {ok}/{len(resultados)}')
    print(f'  ❌ Falhas: {len(resultados) - ok}/{len(resultados)}')


---
## 🔧 CÉLULA 12 — Diagnóstico e Solução de Problemas

In [ ]:
# CÉLULA 12 — DIAGNÓSTICO COMPLETO
# =====================================================================
import sys, os, importlib
sys.path.insert(0, '/content/dark-stickman-studio')

print('=== DARK STICKMAN — Diagnóstico Completo ===')
print()

# Módulos core
modulos = [
    ('core.ffmpeg_utils', 'FFmpeg Utils'),
    ('core.piper_tts', 'Piper TTS'),
    ('core.cloud_tts', 'Cloud TTS'),
    ('core.legenda', 'Legendas SRT'),
    ('core.dark_video_studio', 'Dark Video Studio'),
    ('pipeline.f01_producao', 'Pipeline F01'),
]

print('📦 Módulos:')
for mod_name, label in modulos:
    try:
        importlib.import_module(mod_name)
        print(f'  ✅ {label}')
    except Exception as e:
        print(f'  ❌ {label}: {e}')

# TTS
print('\n🎙️  TTS:')
from core.piper_tts import check_tts_engines
engines = check_tts_engines()
for eng, ok in engines.items():
    print(f'  {"✅" if ok else "❌"} {eng}')

# Cloud TTS
from core.cloud_tts import check_cloud_tts
cloud = check_cloud_tts()
print('\n☁️  Cloud TTS:')
for name, info in cloud.items():
    print(f'  {"✅" if info["available"] else "❌"} {name}: {info["note"]}')

# Video
print('\n🎬 Video Backends:')
from core.dark_video_studio import check_backends
backends = check_backends()
for name, info in backends.items():
    print(f'  {"✅" if info.get("available") else "❌"} {name}: {info.get("note", "")}')

print('\n✅ Diagnóstico concluído!')

---
## 📚 Guia Rápido

| Célula | Função | Obrigatória |
|--------|--------|-------------|
| 1 | Diagnóstico de hardware | ✅ Sempre |
| 2 | Instalação de dependências | ✅ Primeira vez |
| 3 | Clone do repositório | ✅ Sempre |
| 4 | Chaves de API | ⚡ Para melhores vozes |
| 5 | Download Piper TTS | ⚡ Para voz local |
| 6 | **Configure seu roteiro** | ✅ Sempre |
| 7 | **Executa o pipeline** | ✅ Sempre |
| 8 | Download do vídeo | ✅ Para baixar |
| 9 | Vídeo realista (GPU) | 🎮 Opcional |
| 10 | Teste TTS | 🔧 Debug |
| 11 | Batch 12 canais | 🏭 Produção |
| 12 | Diagnóstico | 🔧 Solução de problemas |

---

### 🆓 Serviços Gratuitos Usados:
- **Kokoro TTS** — 82M params, PT-BR, sem GPU, sem internet
- **HF Inference API** — MMS-TTS-POR (PT-BR nativo), 30k req/mês
- **ElevenLabs** — 10.000 chars/mês gratuitos (melhor qualidade)
- **Kokoro HF Space** — Gradio público, sem token
- **FFmpeg** — fallback sempre disponível
- **Piper TTS** — modelo local download único

---

> **Aviso**: Todo conteúdo produzido é identificado com `ai_generated: True` e `ai_disclosure: Produzido com DARK STUDIO AAA — DARK CREATIVE LAB` conforme exigido pelas diretrizes do projeto.
>
> **Licença**: MIT — DARK CREATIVE LAB — Não copie código proprietário, modelos privados ou marcas protegidas.